# 02 — Predict trên Spark

> ⚠️ Notebook này viết cho môi trường **có Spark** (Jupyter trên cluster / spark local có Java).
> Không chạy trong CI local. Cài thêm: `uv sync --extra spark`.

Luồng: config → Spark session → đọc features → load `ModelBundle` (từ notebook 01) → predict (pandas UDF) →
quy đổi điểm → SHAP top features → ghi parquet.

In [ ]:
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

from train_model.config import load_config
from train_model.modeling import ModelBundle
from train_model.spark.predict import predict_spark
from train_model.spark.scoring import proba_to_score_spark
from train_model.spark.session import create_spark_session, s3a_conf_from_env
from train_model.spark.shap import top_features_shap_spark

cfg = load_config(ROOT / "configs" / "model_config.yaml")
bundle = ModelBundle.load(ROOT / cfg.paths["artifacts_dir"] / cfg.model.model_code)
print(bundle.model_code, bundle.features, bundle.score_bounds)

## Spark session

Cấu hình cluster (k8s image, memory, ...) đặt ở `spark.conf` trong config.
Key S3 đọc từ biến môi trường `S3_ENDPOINT`, `S3_ACCESS_KEY`, `S3_SECRET_KEY`.

In [ ]:
conf = {**cfg.spark.get("conf", {}), **s3a_conf_from_env()}
spark = create_spark_session(cfg.spark["app_name"], master=cfg.spark.get("master"), conf=conf)
spark

## Đọc data

Data thật: parquet ở `paths.predict_input` (vd `s3a://<bucket>/features/merged/date=20250801`).
Để thử nhanh không có data, đặt `USE_DEMO = True` để tạo data giả.

In [ ]:
USE_DEMO = False
RUN_DATE = "20250801"

if USE_DEMO:
    from train_model.data import make_synthetic_dataset

    sdf = spark.createDataFrame(make_synthetic_dataset(n_rows=5000))
else:
    sdf = spark.read.parquet(f"{cfg.paths['predict_input']}/date={RUN_DATE}")
sdf.printSchema()

## Predict + quy đổi điểm

In [ ]:
sc = cfg.scoring
pred = predict_spark(spark, bundle, sdf, id_col=cfg.model.id_col)
pred = proba_to_score_spark(
    pred, target_label=sc["target_label"], lower=sc["lower"], upper=sc["upper"],
    pct_lb=sc["pct_lb"], pct_ub=sc["pct_ub"], exponent=sc["exponent"], bounds=bundle.score_bounds,
)
pred.show(10)
pred.describe("predict_proba", "score").show()

## SHAP top features

In [ ]:
shap_df = top_features_shap_spark(spark, bundle, sdf, top_n=10, sample_size=20000)
shap_df.show()

## Ghi kết quả

In [ ]:
out = cfg.paths["predict_output"]
pred.write.mode("overwrite").parquet(f"{out}/predict/date={RUN_DATE}")
shap_df.write.mode("overwrite").parquet(f"{out}/shap_value/date={RUN_DATE}")
print("written to", out)